#### Import the required libraries:

In [47]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, roc_curve
import plotly.express as px
import plotly.graph_objects as go

from pathlib import Path
import joblib

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = models.resnet50(weights='IMAGENET1K_V1')

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 1)
model = model.to(device)
print(model.fc)

Using device: cuda
Linear(in_features=2048, out_features=1, bias=True)


### Load the dataset:

In [17]:
IMG_DIR = (r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\raw_images")
train_df = pd.read_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\train.csv")
val_df = pd.read_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\val.csv")
test_df = pd.read_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\test.csv")

# Data Preprocessing:

In [18]:
class DeepfakeDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)                                # how many rows/images total

    def __getitem__(self, idx):
        row = self.df.iloc[idx]                            # get the row at position idx
        img_path = f"{self.img_dir}/{row['image_id']}.jpg"
        image = Image.open(img_path).convert("RGB")        # open image, force RGB (avoid grayscale/RGBA issues)

        label = 1 if row['label'] == 'FAKE' else 0         # convert text label to number (models need numbers, not text)

        if self.transform:
            image = self.transform(image)                  # apply resize/normalize/augmentation

        return image, label

### Transform Compose:

Red channel: mean = 0.485, std = 0.229

Green channel: mean = 0.456, std = 0.224

Blue channel: mean = 0.406, std = 0.225

### Train:

In [19]:
train_transform = transforms.Compose([
                                        transforms.Resize((224, 224)),
                                        transforms.RandomHorizontalFlip(),
                                        transforms.RandomRotation(10),
                                        transforms.ToTensor(),
                                        transforms.Normalize( mean=[0.485, 0.456, 0.406],
                                                              std=[0.229, 0.224, 0.225])
                                    ])

### Test/Val:

In [20]:
val_test_transform = transforms.Compose([
                                transforms.Resize((224, 224)),
                                transforms.ToTensor(),
                                transforms.Normalize( mean=[0.485, 0.456, 0.406],
                                                      std=[0.229, 0.224, 0.225])
                                        ])

In [21]:
train_dataset = DeepfakeDataset(dataframe= train_df, img_dir= IMG_DIR, transform= train_transform)
val_dataset = DeepfakeDataset(dataframe= val_df, img_dir= IMG_DIR, transform= val_test_transform)
test_dataset = DeepfakeDataset(dataframe= test_df, img_dir= IMG_DIR, transform= val_test_transform)

In [22]:
image, label = train_dataset[0]
print("Label:", label)
print("Image type:", type(image))
print("Image shape:", image.shape)

Label: 1
Image type: <class 'torch.Tensor'>
Image shape: torch.Size([3, 224, 224])


# Data Loader:

In [23]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [24]:
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 122
Val batches: 26
Test batches: 27


In [25]:
images, labels = next(iter(train_loader))
print("Batch of images shape:", images.shape)
print("Batch of labels shape:", labels.shape)

Batch of images shape: torch.Size([32, 3, 224, 224])
Batch of labels shape: torch.Size([32])


# Model Building:

### Model -1 - CNN Model:

In [26]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.block5 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.dropout_conv = nn.Dropout(0.3)
        self.gap = nn.AdaptiveAvgPool2d(1)   # Global Average Pooling -> output size 1x1

        self.fc1 = nn.Linear(512, 256)
        self.dropout_fc = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, 1)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = self.dropout_conv(x)

        x = self.gap(x)             # shape: [batch, 512, 1, 1]
        x = x.view(x.size(0), -1)   # flatten -> [batch, 512]

        x = torch.relu(self.fc1(x))
        x = self.dropout_fc(x)
        x = self.fc2(x)             # raw output, sigmoid applied later via loss function
        return x

cnn_model = CNN().to(device)
print(cnn_model)

CNN(
  (block1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block4): Sequential(
    (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), 

### Model - 2 - Transfer Learning (ResNet 50):

In [27]:
resnet_model = models.resnet50(weights='IMAGENET1K_V1')

# Freeze all layers first
for param in resnet_model.parameters():
    param.requires_grad = False

# Unfreeze the last block (layer4) + replace final classifier head
for param in resnet_model.layer4.parameters():
    param.requires_grad = True

resnet_model.fc = nn.Sequential(
                    nn.Linear(resnet_model.fc.in_features, 256),
                    nn.ReLU(),
                    nn.Dropout(0.5),
                    nn.Linear(256, 1)
)

resnet_model = resnet_model.to(device)
print(resnet_model.fc)

Sequential(
  (0): Linear(in_features=2048, out_features=256, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=256, out_features=1, bias=True)
)


### Model - 3 - Vision Transformer (ViT):


In [28]:
vit_model = models.vit_b_16(weights='IMAGENET1K_V1')

# Freeze all layers first
for param in vit_model.parameters():
    param.requires_grad = False

# Unfreeze the last transformer encoder block for fine-tuning
for param in vit_model.encoder.layers[-1].parameters():
    param.requires_grad = True

# Replace classification head
vit_model.heads = nn.Sequential(
    nn.Linear(vit_model.heads[0].in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 1)
)

vit_model = vit_model.to(device)
print(vit_model.heads)

Sequential(
  (0): Linear(in_features=768, out_features=256, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=256, out_features=1, bias=True)
)


# Model Training:

In [29]:
def train_model(model, optimizer, num_epochs):
    
    history = {
        "train_acc": [],
        "val_acc": [],
        "train_loss": []
    }

    for epoch in range(num_epochs):

        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1}/{num_epochs}"
        ):
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            preds = (torch.sigmoid(outputs) > 0.5).float()

            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_acc = correct / total
        train_loss = running_loss / len(train_loader)

        history["train_acc"].append(train_acc)
        history["train_loss"].append(train_loss)

        # Validation
        model.eval()
        val_correct, val_total = 0, 0

        with torch.no_grad():
            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.float().unsqueeze(1).to(device)

                outputs = model(images)

                preds = (torch.sigmoid(outputs) > 0.5).float()

                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_acc = val_correct / val_total

        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch + 1}: "
            f"Loss={train_loss:.4f}, "
            f"Train Acc={train_acc:.4f}, "
            f"Val Acc={val_acc:.4f}"
        )

    return history

### CNN Model:

In [30]:
criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
                        cnn_model.parameters(),
                        lr=0.001
)

cnn_history = train_model(
                        cnn_model,
                        optimizer,
                        num_epochs=15
)


Epoch 1/15: 100%|██████████| 122/122 [01:23<00:00,  1.46it/s]


Epoch 1: Loss=0.1842, Train Acc=0.9393, Val Acc=0.9483


Epoch 2/15: 100%|██████████| 122/122 [01:15<00:00,  1.61it/s]


Epoch 2: Loss=0.1518, Train Acc=0.9486, Val Acc=0.9327


Epoch 3/15: 100%|██████████| 122/122 [01:16<00:00,  1.60it/s]


...

Epoch 13: Loss=0.1125, Train Acc=0.9597, Val Acc=0.9663


Epoch 14/15: 100%|██████████| 122/122 [01:17<00:00,  1.57it/s]


Epoch 14: Loss=0.1083, Train Acc=0.9643, Val Acc=0.9567


Epoch 15/15: 100%|██████████| 122/122 [01:17<00:00,  1.58it/s]


Epoch 15: Loss=0.1027, Train Acc=0.9638, Val Acc=0.9615


### ResNet 50:

In [31]:
criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam([
                        {"params": resnet_model.layer4.parameters(), "lr": 1e-5},
                        {"params": resnet_model.fc.parameters(), "lr": 1e-3}
])

resnet_history = train_model(
                                resnet_model,
                                optimizer,
                                num_epochs=10
)

Epoch 1/10: 100%|██████████| 122/122 [01:27<00:00,  1.40it/s]


Epoch 1: Loss=0.1029, Train Acc=0.9674, Val Acc=0.9892


Epoch 2/10: 100%|██████████| 122/122 [01:27<00:00,  1.39it/s]


Epoch 2: Loss=0.0274, Train Acc=0.9910, Val Acc=0.9916


Epoch 3/10: 100%|██████████| 122/122 [01:25<00:00,  1.42it/s]


...

Epoch 8: Loss=0.0030, Train Acc=0.9987, Val Acc=0.9940


Epoch 9/10: 100%|██████████| 122/122 [01:25<00:00,  1.42it/s]


Epoch 9: Loss=0.0061, Train Acc=0.9979, Val Acc=0.9928


Epoch 10/10: 100%|██████████| 122/122 [01:24<00:00,  1.44it/s]


Epoch 10: Loss=0.0056, Train Acc=0.9985, Val Acc=0.9928


### ViT:

In [32]:
criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam([
                        {"params": vit_model.encoder.layers[-1].parameters(), "lr": 1e-5},
                        {"params": vit_model.heads.parameters(), "lr": 1e-3}
])

vit_history = train_model(
                            vit_model,
                            optimizer,
                            num_epochs=10
)

Epoch 1/10: 100%|██████████| 122/122 [02:55<00:00,  1.44s/it]


Epoch 1: Loss=0.0723, Train Acc=0.9782, Val Acc=0.9880


Epoch 2/10: 100%|██████████| 122/122 [02:55<00:00,  1.44s/it]


Epoch 2: Loss=0.0251, Train Acc=0.9936, Val Acc=0.9916


Epoch 3/10: 100%|██████████| 122/122 [02:53<00:00,  1.43s/it]


...

Epoch 8: Loss=0.0017, Train Acc=0.9992, Val Acc=0.9916


Epoch 9/10: 100%|██████████| 122/122 [02:54<00:00,  1.43s/it]


Epoch 9: Loss=0.0031, Train Acc=0.9992, Val Acc=0.9880


Epoch 10/10: 100%|██████████| 122/122 [02:55<00:00,  1.44s/it]


Epoch 10: Loss=0.0026, Train Acc=0.9987, Val Acc=0.9928


# Model Evaluation:

In [33]:
def evaluate_model(model, loader, dataset_name="Set"):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_preds = np.array(all_preds).flatten()
    all_labels = np.array(all_labels).flatten()
    all_probs = np.array(all_probs).flatten()

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)

    print(f"--- {dataset_name} Metrics ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"AUC-ROC:   {auc:.4f}\n")

    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "auc": auc,
            "labels": all_labels, "preds": all_preds, "probs": all_probs}

### CNN Model Result:

In [34]:
print("===== CUSTOM CNN RESULTS =====\n")
cnn_train_results = evaluate_model(cnn_model, train_loader, "Train")
cnn_test_results = evaluate_model(cnn_model, test_loader, "Test")

===== CUSTOM CNN RESULTS =====

--- Train Metrics ---
Accuracy:  0.9725
Precision: 0.9731
Recall:    0.9716
F1-Score:  0.9724
AUC-ROC:   0.9955

--- Test Metrics ---
Accuracy:  0.9652
Precision: 0.9531
Recall:    0.9783
F1-Score:  0.9655
AUC-ROC:   0.9959



### ResNet 50 Model Result:

In [35]:
print("===== RESNET50 RESULTS =====\n")
resnet_train_results = evaluate_model(resnet_model, train_loader, "Train")
resnet_test_results = evaluate_model(resnet_model, test_loader, "Test")

===== RESNET50 RESULTS =====

--- Train Metrics ---
Accuracy:  0.9997
Precision: 1.0000
Recall:    0.9995
F1-Score:  0.9997
AUC-ROC:   1.0000

--- Test Metrics ---
Accuracy:  0.9928
Precision: 0.9952
Recall:    0.9904
F1-Score:  0.9928
AUC-ROC:   0.9998



### ViT Model Result:

In [36]:
print("===== VIT RESULTS =====\n")
vit_train_results = evaluate_model(vit_model, train_loader, "Train")
vit_test_results = evaluate_model(vit_model, test_loader, "Test")

===== VIT RESULTS =====

--- Train Metrics ---
Accuracy:  0.9997
Precision: 1.0000
Recall:    0.9995
F1-Score:  0.9997
AUC-ROC:   1.0000

--- Test Metrics ---
Accuracy:  0.9916
Precision: 0.9951
Recall:    0.9880
F1-Score:  0.9915
AUC-ROC:   0.9986



### Result Summary:

In [37]:
def create_results_summary(train_results, test_results):
    
    return pd.DataFrame({
        "Metric": ["Accuracy", "Precision", "Recall", "F1-Score", "AUC-ROC"],
        
        "Train": [
                    train_results["accuracy"],
                    train_results["precision"],
                    train_results["recall"],
                    train_results["f1"],
                    train_results["auc"]
        ],

        "Test": [
                    test_results["accuracy"],
                    test_results["precision"],
                    test_results["recall"],
                    test_results["f1"],
                    test_results["auc"]
        ]
    })

In [38]:
cnn_results_summary = create_results_summary(
                                            cnn_train_results,
                                            cnn_test_results
                                            )

resnet_results_summary = create_results_summary(
                                                resnet_train_results,
                                                resnet_test_results
                                                )

vit_results_summary = create_results_summary(
                                            vit_train_results,
                                            vit_test_results
                                            )


In [39]:
print("===== CUSTOM CNN RESULTS =====\n")
print(cnn_results_summary)

print("\n===== RESNET50 RESULTS =====\n")
print(resnet_results_summary)

print("\n===== VIT RESULTS =====\n")
print(vit_results_summary)

===== CUSTOM CNN RESULTS =====

      Metric     Train      Test
0   Accuracy  0.972501  0.965228
1  Precision  0.973127  0.953052
2     Recall  0.971620  0.978313
3   F1-Score  0.972373  0.965517
4    AUC-ROC  0.995502  0.995877

===== RESNET50 RESULTS =====

      Metric     Train      Test
0   Accuracy  0.999743  0.992806
1  Precision  1.000000  0.995157
2     Recall  0.999484  0.990361
3   F1-Score  0.999742  0.992754
4    AUC-ROC  0.999999  0.999822

===== VIT RESULTS =====

      Metric     Train      Test
0   Accuracy  0.999743  0.991607
1  Precision  1.000000  0.995146
2     Recall  0.999484  0.987952
3   F1-Score  0.999742  0.991536
4    AUC-ROC  1.000000  0.998626


# Confusion Matrix:

### CNN Model:

In [50]:
# Train Confusion Matrix
cm_cnn_train = confusion_matrix(
                              cnn_train_results["labels"],
                              cnn_train_results["preds"]
                            )

cnn_train_fig = px.imshow(
                        cm_cnn_train,
                        text_auto=True,
                        x=['REAL', 'FAKE'],
                        y=['REAL', 'FAKE'],
                        color_continuous_scale='blues',
                        title='Confusion Matrix for CNN Model - Train'
              )

cnn_train_fig.update_layout(
                            height = 500, width = 500,
                            xaxis_title='Predicted',
                            yaxis_title='Actual'
                  )
cnn_train_fig.show()


# Test Confusion Matrix
cm_cnn_test = confusion_matrix(
                                cnn_test_results["labels"],
                                cnn_test_results["preds"]
                          )

cnn_test_fig = px.imshow(
                        cm_cnn_test,
                        text_auto=True,
                        x=['REAL', 'FAKE'],
                        y=['REAL', 'FAKE'],
                        color_continuous_scale='Blues',
                        title='Confusion Matrix for CNN Model - Test'
)

cnn_test_fig.update_layout( height = 500, width = 500,
                            xaxis_title='Predicted',
                            yaxis_title='Actual'
)

cnn_test_fig.show()

### ResNet Model:

In [51]:
# Train Confusion Matrix
cm_resnet_train = confusion_matrix(
                        resnet_train_results["labels"],
                        resnet_train_results["preds"]
)

resnet_train_fig = px.imshow(
                            cm_resnet_train,
                            text_auto=True,
                            x=['REAL', 'FAKE'],
                            y=['REAL', 'FAKE'],
                            color_continuous_scale='Greens',
                            title='Confusion Matrix - Train (ResNet50)'
)

resnet_train_fig.update_layout( height = 500, width = 500,
                                xaxis_title='Predicted',
                                yaxis_title='Actual'
)

resnet_train_fig.show()

joblib.dump(resnet_train_fig, r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\confu_resnet_train_fig.pkl")
print(" confusion resnet_train_fig saved as pkl file")



# Test Confusion Matrix
cm_resnet_test = confusion_matrix(
                          resnet_test_results["labels"],
                          resnet_test_results["preds"]
)

resnet_test_fig = px.imshow(
                            cm_resnet_test,
                            text_auto=True,
                            x=['REAL', 'FAKE'],
                            y=['REAL', 'FAKE'],
                            color_continuous_scale='Greens',
                            title='Confusion Matrix - Test (ResNet50)'
)

resnet_test_fig.update_layout( height = 500, width = 500,
                                xaxis_title='Predicted',
                                yaxis_title='Actual'
)

resnet_test_fig.show()

joblib.dump(resnet_test_fig, r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\confu_resnet_test_fig.pkl")
print(" confusion resnet_test_fig saved as pkl file")

 confusion resnet_train_fig saved as pkl file


 confusion resnet_test_fig saved as pkl file


### ViT Model:

In [42]:
# Train Confusion Matrix
cm_vit_train = confusion_matrix(
                        vit_train_results["labels"],
                        vit_train_results["preds"]
)

vit_train_fig = px.imshow(
                cm_vit_train,
                text_auto=True,
                x=['REAL', 'FAKE'],
                y=['REAL', 'FAKE'],
                color_continuous_scale='Oranges',
                title='Confusion Matrix - Train (ViT)'
)

vit_train_fig.update_layout( height = 500, width = 700,
                  xaxis_title='Predicted',
                  yaxis_title='Actual'
)

vit_train_fig.show()


# Test Confusion Matrix
cm_vit_test = confusion_matrix(
                          vit_test_results["labels"],
                          vit_test_results["preds"]
)

vit_test_fig = px.imshow(
                        cm_vit_test,
                        text_auto=True,
                        x=['REAL', 'FAKE'],
                        y=['REAL', 'FAKE'],
                        color_continuous_scale='Oranges',
                        title='Confusion Matrix - Test (ViT)'
)

vit_test_fig.update_layout( height = 500, width = 700,
                            xaxis_title='Predicted',
                            yaxis_title='Actual'
)

vit_test_fig.show()

# ROC Curve:

### CNN Model:

In [43]:
fpr_tr, tpr_tr, _ = roc_curve(
                    cnn_train_results["labels"],
                    cnn_train_results["probs"]
)

fpr_te, tpr_te, _ = roc_curve(
                      cnn_test_results["labels"],
                      cnn_test_results["probs"]
)

fig = go.Figure()

fig.add_scatter(
                  x=fpr_tr,
                  y=tpr_tr,
                  mode='lines',
                  name=f'Train (AUC={cnn_train_results["auc"]:.3f})'
                )

fig.add_scatter(
                x=fpr_te,
                y=tpr_te,
                mode='lines',
                name=f'Test (AUC={cnn_test_results["auc"]:.3f})'
                )

fig.add_scatter(
                x=[0, 1],
                y=[0, 1],
                mode='lines',
                line=dict(dash='dash'),
                name='Random'
                )

fig.update_layout( height = 500, width = 800,
                  title='ROC Curve - Custom CNN',
                  xaxis_title='False Positive Rate',
                  yaxis_title='True Positive Rate'
                  )

fig.show()

### ResNet 50 Model:

In [53]:
fpr_tr, tpr_tr, _ = roc_curve(
                          resnet_train_results["labels"],
                          resnet_train_results["probs"]
)

fpr_te, tpr_te, _ = roc_curve(
                          resnet_test_results["labels"],
                          resnet_test_results["probs"]
)

fig = go.Figure()

fig.add_scatter(
              x=fpr_tr,
              y=tpr_tr,
              mode='lines',
              name=f'Train (AUC={resnet_train_results["auc"]:.3f})'
)

fig.add_scatter(
              x=fpr_te,
              y=tpr_te,
              mode='lines',
              name=f'Test (AUC={resnet_test_results["auc"]:.3f})'
)

fig.add_scatter(
                x=[0, 1],
                y=[0, 1],
                mode='lines',
                line=dict(dash='dash'),
                name='Random'
)

fig.update_layout( height = 500, width = 800,
                  title='ROC Curve - ResNet50',
                  xaxis_title='False Positive Rate',
                  yaxis_title='True Positive Rate'
)

fig.show()

joblib.dump(fig, r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\resnet_roc_curve_fig.pkl")
print(" ROC Curve - ResNet50 saved as pkl file")

 ROC Curve - ResNet50 saved as pkl file


### ViT Model:

In [45]:
fpr_tr, tpr_tr, _ = roc_curve(
                              vit_train_results["labels"],
                              vit_train_results["probs"]
)

fpr_te, tpr_te, _ = roc_curve(
                              vit_test_results["labels"],
                              vit_test_results["probs"]
)

fig = go.Figure()

fig.add_scatter(
                x=fpr_tr,
                y=tpr_tr,
                mode='lines',
                name=f'Train (AUC={vit_train_results["auc"]:.3f})'
)

fig.add_scatter(
                x=fpr_te,
                y=tpr_te,
                mode='lines',
                name=f'Test (AUC={vit_test_results["auc"]:.3f})'
)

fig.add_scatter(
                x=[0, 1],
                y=[0, 1],
                mode='lines',
                line=dict(dash='dash'),
                name='Random'
)

fig.update_layout( height = 500, width = 700,
                  title='ROC Curve - ViT',
                  xaxis_title='False Positive Rate',
                  yaxis_title='True Positive Rate'
)

fig.show()

# Save the Model:

In [46]:
# Saving the Cnn model:
torch.save( cnn_model.state_dict(),
            r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\custom_cnn_model.pth"
            )
print("Custom CNN saved successfully!")


# Saving the ResNet 50 Model:
torch.save( resnet_model.state_dict(),
            r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\resnet50_model.pth"
            )
print("Transfer Learning ResNet50 Model saved successfully!")


# Saving the ViT Model:
torch.save( vit_model.state_dict(),
            r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\vit_model.pth"
            )
print("ViT Model saved successfully!")


Custom CNN saved successfully!
Transfer Learning ResNet50 Model saved successfully!
ViT Model saved successfully!
